In [8]:
import pyodbc
import pandas as pd
import numpy as np

In [ ]:
conn_172 = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=172.22.24.232;'  # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;'         # Tên cơ sở dữ liệu
    'UID=sa;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)

In [10]:
query_tailieu = """SELECT ID_tai_lieu, Ma_tai_lieu, Ngay_giao_dich 
                    FROM olap.DIM_Tai_lieu"""
df_tai_lieu = pd.read_sql(query_tailieu, conn_172)
print(df_tai_lieu)

C:\Users\phung\AppData\Local\Temp\ipykernel_5820\227603785.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tai_lieu = pd.read_sql(query_tailieu, conn_172)


       ID_tai_lieu       Ma_tai_lieu  Ngay_giao_dich
0                0  (Không xác định)               0
1                7       SK020000011        20070417
2               12       SK020000015        20070417
3               26         SKV000004        20070417
4               30       SK020000030        20070417
...            ...               ...             ...
63526         5880  (Không xác định)               0
63527         5887  (Không xác định)               0
63528         5888  (Không xác định)               0
63529         5896  (Không xác định)               0
63530         5933  (Không xác định)               0

[63531 rows x 3 columns]


In [11]:
query_xepgia = """SELECT ID_xep_gia, ID_tai_lieu 
                    FROM olap.DIM_Xep_gia"""
df_xep_gia = pd.read_sql(query_xepgia, conn_172)
print(df_xep_gia)

C:\Users\phung\AppData\Local\Temp\ipykernel_5820\1171866631.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_xep_gia = pd.read_sql(query_xepgia, conn_172)


        ID_xep_gia  ID_tai_lieu
0                0            0
1           182098           31
2           182099           31
3           182100           31
4           182101           31
...            ...          ...
595055     1046993        68629
595056     1046994        68621
595057     1046995        68630
595058     1046996        68632
595059     1046997        68631

[595060 rows x 2 columns]


In [12]:
so_ban_sach = df_xep_gia.groupby('ID_tai_lieu')['ID_xep_gia'].count().reset_index()
so_ban_sach = so_ban_sach.rename(columns={'ID_xep_gia': 'So_ban_sach'})

In [13]:
# Lấy danh sách ID_tai_lieu có bản sách
tai_lieu_co_ban_sach = df_xep_gia['ID_tai_lieu'].unique()

# Lọc df_tai_lieu để chỉ tính đầu sách cho những tài liệu có bản sách
df_dau_sach = df_tai_lieu[df_tai_lieu['ID_tai_lieu'].isin(tai_lieu_co_ban_sach)]

# Tính số đầu sách
so_dau_sach = df_dau_sach.groupby('Ma_tai_lieu')['ID_tai_lieu'].nunique().reset_index()
so_dau_sach = so_dau_sach.rename(columns={'ID_tai_lieu': 'So_dau_sach'})


In [14]:
df_final = df_tai_lieu.merge(so_ban_sach, on='ID_tai_lieu', how='left')
df_final = df_final.merge(so_dau_sach, on='Ma_tai_lieu', how='left')

# Gán mặc định 0 cho bản sách và đầu sách nếu thiếu
df_final['So_ban_sach'] = df_final['So_ban_sach'].fillna(0).astype(int)
df_final['So_dau_sach'] = df_final['So_dau_sach'].fillna(0).astype(int)

print(df_final)

       ID_tai_lieu       Ma_tai_lieu  Ngay_giao_dich  So_ban_sach  So_dau_sach
0                0  (Không xác định)               0            1            1
1                7       SK020000011        20070417            0            0
2               12       SK020000015        20070417            0            0
3               26         SKV000004        20070417            1            1
4               30       SK020000030        20070417            2            1
...            ...               ...             ...          ...          ...
63526         5880  (Không xác định)               0            0            1
63527         5887  (Không xác định)               0            0            1
63528         5888  (Không xác định)               0            0            1
63529         5896  (Không xác định)               0            0            1
63530         5933  (Không xác định)               0            0            1

[63531 rows x 5 columns]


In [15]:
cursor_dwh = conn_172.cursor()
insert_query = """
                    INSERT INTO olap.FACT_Tai_lieu (ID_tai_lieu,
                                                    Ma_tai_lieu, 
                                                    ID_date, 
                                                    So_dau_sach,
                                                    So_ban_sach)
                    VALUES (?, ?, ?, ?, ?)"""
for index, row in df_final.iterrows():
    # Trích xuất giá trị từ các cột
    values = (row['ID_tai_lieu'],
                row['Ma_tai_lieu'],
                row['Ngay_giao_dich'],
                row['So_dau_sach'],
                row['So_ban_sach']) 
    cursor_dwh.execute(insert_query, values)
conn_172.commit()
conn_172.close()